# Rounding noise floor: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_noisefloor_2026-11-04.md` (N1–N7). All runs are eager and share this session. The 33-frame part takes about 50 min; the optional 81-frame part adds about 26 min (~2.5 compute units in total).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit --format=csv

In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, diffusers.__version__, transformers.__version__)

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## 33 frames, 4 prompts

Reference first (default attention = flash, same code path as the baseline, ~15 min). Then check N1 against the committed hashes of the earlier eager runs.

In [ ]:
!python bench/profile_baseline.py --frames 33 --kernels --out results/flash_33f

In [ ]:
!python bench/hash_frames.py results/flash_33f --check results/eager_33f/frames_sha256.json

Rounding control: memory-efficient attention inside the transformer, same seed (~17 min). Then N2, N3 and N6.

In [ ]:
!python bench/profile_baseline.py --frames 33 --attn efficient --kernels --out results/efficient_33f

In [ ]:
!python bench/compare_frames.py --ref results/flash_33f --test results/efficient_33f

Sample-to-sample scale: flash, seed 1 (~15 min). Then N5.

In [ ]:
!python bench/profile_baseline.py --frames 33 --seed 1 --out results/seed1_33f

In [ ]:
!python bench/compare_frames.py --ref results/flash_33f --test results/seed1_33f

CLIP score for N7 (downloads ViT-L/14, ~1.7 GB).

In [ ]:
!python bench/clip_score.py results/flash_33f results/efficient_33f results/seed1_33f

## 81 frames, fox only (optional)

Reference (~12 min), hash check against the 2026-11-03 baseline, then memory-efficient (~14 min) for N4.

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --out results/flash_81f

In [ ]:
!python bench/hash_frames.py results/flash_81f --check results/base_81f/frames_sha256.json

In [ ]:
!python bench/profile_baseline.py --frames 81 --n-prompts 1 --attn efficient --kernels --out results/efficient_81f

In [ ]:
!python bench/compare_frames.py --ref results/flash_81f --test results/efficient_81f

In [ ]:
# The comparisons, hashes and CLIP scores are computed above, so no .npy is needed locally
!zip -qr noise_results.zip results -x '*.npy' && ls -lh noise_results.zip
from google.colab import files; files.download('noise_results.zip')